In [0]:
pip install faker

In [0]:
from faker import Faker

In [0]:
fake = Faker()
fake.name()
fake.email()
fake.city()
fake.date()
fake.uuid4()

In [0]:
import pandas as pd
import numpy
import os
from datetime import datetime, timedelta
import random
import utils
from datetime import *
date = date.today()

In [0]:
def customer_generator(n=500):
    customers = []
    for i in range(n):
        customers.append({
            "customer_id"      : f"CUST{i+1:05d}",
            "name"             : fake.name(),
            "email"            : fake.email(),
            "phone"            : fake.phone_number(),
            "city"             : random.choice([
                                    "Mumbai","Delhi","Bangalore",
                                    "Chennai","Hyderabad","Pune",
                                    "Kolkata","Ahmedabad"]),
            "state"            : fake.state(),
            "registration_date": fake.date_between(
                                    start_date="-2y",
                                    end_date="today"),
            "customer_segment" : random.choice([
                                    "Regular","Premium","VIP"])
        })
    return pd.DataFrame(customers)
    

In [0]:
def generate_products (n=100):
    categories = {
        "Electronics" : ["Mobile","Laptop","Tablet","TV"],
        "Fashion"     : ["Shirt","Jeans","Dress","Shoes"],
        "Grocery"     : ["Rice","Oil","Sugar","Flour"],
        "Home"        : ["Chair","Table","Sofa","Lamp"]
        }
    products = []
    for i in range(n):
        category    = random.choice(list(categories.keys()))
        sub_category = random.choice(categories[category])
        products.append({
            "product_id"     : f"PROD{i+1:05d}",
            "name"           : f"{sub_category} {fake.word().title()}",
            "category"       : category,
            "sub_category"   : sub_category,
            "brand"          : fake.company(),
            "unit_price"     : round(random.uniform(10, 50000), 2),
            "stock_quantity" : random.randint(0, 1000),
            "supplier"       : fake.company()
        })
    return pd.DataFrame(products)

In [0]:
def generate_orders(customers_df, products_df, n=5000):
    orders = []
    for i in range(n):
        customer = customers_df.sample(1).iloc[0]
        product  = products_df.sample(1).iloc[0]
        quantity = random.randint(1, 10)
        amount   = round(quantity * product["unit_price"], 2)

        orders.append({
            "order_id"      : f"ORD{i+1:06d}",
            "customer_id"   : customer["customer_id"],
            "product_id"    : product["product_id"],
            "order_date"    : datetime.now().strftime("%Y-%m-%d"),
            "quantity"      : quantity,
            "unit_price"    : product["unit_price"],
            "total_amount"  : amount,
            "status"        : random.choice([
                                "Delivered","Pending",
                                "Cancelled","Processing"]),
            "payment_method": random.choice([
                                "UPI","Credit Card",
                                "Debit Card","NetBanking",
                                "COD"])
        })
    return pd.DataFrame(orders)

In [0]:
def generate_payments(orders_df):
    payments = []
    for i, order in orders_df.iterrows():
        payments.append({
            "payment_id"    : f"PAY{i+1:06d}",
            "order_id"      : order["order_id"],
            "payment_date"  : datetime.now().strftime("%Y-%m-%d"),
            "payment_method": order["payment_method"],
            "amount"        : order["total_amount"],
            "status"        : random.choice([
                                "Success","Failed","Pending"]),
            "transaction_id": fake.uuid4()
        })
    return pd.DataFrame(payments)

In [0]:
from utils.constants import (
    AWS_ACCESS_KEY_ID,
    AWS_SECRET_ACCESS_KEY,
    BUCKET,
    REGION
    
)

In [0]:
import boto3
import io

def upload_to_s3(df, bucket, key):
    s3 = boto3.client(
        "s3",
        aws_access_key_id=AWS_ACCESS_KEY_ID,
        aws_secret_access_key=AWS_SECRET_ACCESS_KEY,
        region_name=REGION
    )

    csv_buffer = io.StringIO()
    df.to_csv(csv_buffer, index=False)

    s3.put_object(
        Bucket=bucket,
        Key=key,
        Body=csv_buffer.getvalue()
    )

    print(f"✅ Uploaded to s3://{bucket}/{key}")

    

In [0]:
def generate_daily_data():
    customers_df = customer_generator()
    products_df = generate_products()
    orders_df = generate_orders(customers_df, products_df,5000)
    payments_df = generate_payments(orders_df)
    
     # Upload to S3
    upload_to_s3(customers_df, BUCKET,
        f"Input_Data/customers/date={date}/customers.csv")
    upload_to_s3(products_df,  BUCKET,
        f"Input_Data/products/date={date}/products.csv")
    upload_to_s3(orders_df,    BUCKET,
        f"Input_Data/orders/date={date}/orders.csv")
    upload_to_s3(payments_df,  BUCKET,
        f"Input_Data/payments/date={date}/payments.csv")
  
    
    print(f"✅ Daily data generation complete!")
    print(f"   Customers : {len(customers_df)}")
    print(f"   Products  : {len(products_df)}")
    print(f"   Orders    : {len(orders_df)}")
    print(f"   Payments  : {len(payments_df)}")

generate_daily_data()